We have implemented checkpoint saving and continuing decomposition from a checkpoint.
We will try this here

In [ ]:
import tensorly as tl
import pytensorlab as ptl

from tensorly.tenalg import mode_dot

from tensormet.utils import ThreadBudget, DATA_DIR, compute_num_threads, select_gpu, readonly_dispatch
from tensormet.tucker_tensor import SparseTupleTensor, TuckerDecomposition
from tensormet.config import ExperimentConfig, TrainingConfig, EvalConfig, RunConfig
from tensormet.similarity import load_eval_sentences_cached

from memory_profiler import profile
import os
import pickle
from pathlib import Path
import time
import numpy as np

device = select_gpu()

tl.set_backend("cupy")
dataset = "fineweb-en"
dim = 4000
method = "siiSoftPlus"
divergence = "fr"
name = "test"
random_state = 1
rank = [150, 150, 150]
max_cpu_frac = 0.66
iters = 500
tol = 1e-5
patience = 5
rec_check_every = 1
rec_log_every = 1
sem_check_every = 2
sem_error_type = "absolute_prob_score"
max_cpu_frac = 0.66
thread_budget = ThreadBudget(n_threads=compute_num_threads(max_cpu_frac))

In [1]:
vocab_path = os.path.join(DATA_DIR, "tensors", dataset, f"vocabularies/{dim}.pkl")
with open(vocab_path, "rb") as f:
    vocab = pickle.load(f)
cfg = RunConfig(
    exp=ExperimentConfig(
        dataset=dataset,
        method=method,
        divergence=divergence,
        dim=dim,
        rank=tuple(rank),
        name=name,
        random_state=random_state,
        max_cpu_frac=max_cpu_frac,
        data_dir=Path(DATA_DIR),
    ),
    train=TrainingConfig(
        n_iter_max=iters,
        tol=tol,
        epsilon=1e-12,
        warmup_steps=1 if divergence == "kl" else 50,
        patience=patience,
        normalize_factors=False,
        verbose=True,
        return_errors="full",
    ),
    eval=EvalConfig(
        rec_log_every=rec_log_every,
        rec_check_every=rec_check_every,
        sem_check_every=sem_check_every,
        sem_error_type=sem_error_type,
        remove_OOV=False,
    ),
)
print(cfg)
paths = cfg.artifact_paths()
for p in paths.values():
    if isinstance(p, Path):
        p.parent.mkdir(parents=True, exist_ok=True)



NameError: name 'os' is not defined

In [16]:
sparse_tensor = SparseTupleTensor.load_from_disk(
    dataset=dataset,
    method=method,
    dims=dim,
    map_location="cpu",
)
sparse_tensor.tensor_to_sparse("cupy")

In [17]:
tucker = TuckerDecomposition.load_from_disk(
    dataset=dataset,
    method=method,
    dims=dim,
    rank=150,
    name=name,
    iterations=iters,
)

In [18]:
path_to_vectors = readonly_dispatch(DATA_DIR/ "vectors" / "fineweb_english_vectors.csv")
sentence_sample = load_eval_sentences_cached(vector_path=path_to_vectors,
                                             dataset=dataset,
                                             seed=random_state,
                                             n_samples=10_000,
                                             )

In [20]:
sparse_tensor.non_negative_tucker_with_similarity(
    cfg=cfg,
    thread_budget=thread_budget,
    vocab=vocab,
    sample_sentences=sentence_sample,
    checkpoint_tensor=tucker
)

1: reconstruction error=0.8601154685020447 (Δ=+5.279e-04), time=6.260254859924316
Average expected role vector rank score over 10000 samples: 0.0807505463957858, Average prob score: 0.03692052513360977
Without 4143 OOV: rank 0.13787014921595664, prob 0.06303658336400986
New best semantic score; saving current best core and factors.
2: reconstruction error=0.859599769115448 (Δ=+5.157e-04), time=6.323675870895386
3: reconstruction error=0.8590975999832153 (Δ=+5.022e-04), time=6.2934651374816895
Average expected role vector rank score over 10000 samples: 0.08082268717957383, Average prob score: 0.036860059946775436
Without 4143 OOV: rank 0.13799331941194098, prob 0.06293334066867828
	No semantic improvement: 1/5 (Δ=-1.032e-04)
4: reconstruction error=0.858608067035675 (Δ=+4.895e-04), time=6.295599699020386
5: reconstruction error=0.858130931854248 (Δ=+4.771e-04), time=6.2743799686431885


KeyboardInterrupt: 